### Load libraries

In [ ]:
from scipy import stats
import numpy as np
import os
import pandas as pd
from scipy.stats import kruskal
import scikit_posthocs as sp
import matplotlib.pyplot as plt
import seaborn as sns
import json
from scipy.stats import wilcoxon 
import scipy.stats as stats
from matplotlib.ticker import MultipleLocator
import statsmodels.formula.api as smf
from statannotations.Annotator import Annotator

## Load images and Filter

First load the reults file from the analysis. If an analysis file exists for each sample or technical replicate, the need to be combined into one file. Here, 3 biological replicates are loaded.

In [ ]:
df_4_raw = pd.read_csv(r"YOUR_PATH.csv")
df_5_raw = pd.read_csv(r"YOUR_PATH.csv")
df_6_raw = pd.read_csv(r"YOUR_PATH.csv")

Only cells with at least one Nanorod vesicle are used

In [ ]:
min_vesicle = 0
exp_4 = df_4_raw[(df_4_raw["NR vesicle count"] > min_vesicle) ]
exp_5 = df_5_raw[(df_5_raw["NR vesicle count"] > min_vesicle) ]
exp_6 = df_6_raw[(df_6_raw["NR vesicle count"] > min_vesicle) ]

## Rearrange Dataframes

Before combining all biological replicates into one dataframe, they each need a new column with a biorep number

In [ ]:
# make a list to iterate through the experiments
experiments = [exp_4, exp_5, exp_6]

for e in experiments:
    # Set the index to "cycle". It contains all the information about the sample type and replicate
    e.set_index("cycle", drop = False, inplace=True)

    # We need to define the sample, treatment and technical replicate
    e["sample"] = ""
    e["Benzonase"] = ""
    e["Techn_repl"] = ""
    
    # Extract from the name that was given during microscopy and is contained in "cycle"
    e.loc[e.index.str.contains("NR0"), "sample"] = "NR0"
    e.loc[e.index.str.contains("NR7"), "sample"] = "NR7"
    e.loc[e.index.str.contains("ctrl"), "sample"] = "ctrl"

    e.loc[e.index.str.contains("plus"), "Benzonase"] = "plus"
    e.loc[e.index.str.contains("minus"), "Benzonase"] = "minus"
    e.loc[e.index.str.contains("ctrl"), "Benzonase"] = "minus"

    e.loc[e.index.str.contains("tr1"), "Techn_repl"] = 1
    e.loc[e.index.str.contains("tr2"), "Techn_repl"] = 2
    e.loc[e.index.str.contains("tr3"), "Techn_repl"] = 3

    e["group"] = (e["sample"] + " " + e["Benzonase"])

In [ ]:
# Reset the index
for e in experiments:
    e.reset_index(drop=True, inplace=True)
    e["cell_uid"] = np.arange(len(e)) # add a unique cell ID for each cell per experiment. This is needed to assign overlap values later
    print(e.index.is_unique)

## Extract Data

Now the data from the jason list in the dataframe can be extracted. We use the lists inside of the large dataframe to extract new ones that have information about endosomal and lysosomal colocalization.

In [ ]:
# First, define new variable to save the information in
overlap_df_endo = []
overlap_df_lyso = []

for e in experiments:
    all_overlaps_endo = []

    # extract data for endosomes
    for _, row in e.iterrows():
        overlaps = json.loads(row["List for coloc per vesicle endo"])

        # add information to know where the extracted data comes from
        for overlap in overlaps:
            overlap["Cell ID"] = row["Cell ID"]
            overlap["frame"] = row["frame"]
            overlap["cycle"] = row["cycle"]
            overlap["group"] =row["group"]
            overlap["Techn_repl"] =row["Techn_repl"]
            overlap["cell_uid"] =row["cell_uid"]
    
            all_overlaps_endo.append(overlap)
    
    overlap_df_endo.append(pd.DataFrame(all_overlaps_endo))
    
    # same thing for lysosomes  
    all_overlaps_lyso = []
    
    for _, row in e.iterrows():
        overlaps = json.loads(row["List for coloc per vesicle lyso"])
    
        for overlap in overlaps:
            overlap["Cell ID"] = row["Cell ID"]
            overlap["frame"] = row["frame"]
            overlap["cycle"] = row["cycle"]
            overlap["group"] =row["group"]
            overlap["Techn_repl"] =row["Techn_repl"]
            overlap["cell_uid"] =row["cell_uid"]
    
            all_overlaps_lyso.append(overlap)
    
    overlap_df_lyso.append(pd.DataFrame(all_overlaps_lyso))

Now we can define a threshold. All vesicles that share more area than the threshold will be counted as overlapping. Setting the threshold to at least 50% would be recommended to avoid a vesicle being counted twice in case it overlaps with more than one vesicle. <br>

The threshold can work in two ways: either the area of the Nanorod is used as reference, or the area of the endosome/lysosome is used as reference. Thus, 4 different parameters can be measured in total.

In [ ]:
# Define threshold
min_overlap=0.5

# define new variables for applying the filter in each direction for endosomes and lysosomes
overlap_endo_filtered_nr = []
overlap_endo_filtered_endo = []
overlap_lyso_filtered_nr = []
overlap_lyso_filtered_lyso = []

for df in overlap_df_endo:
    
    overlap_endo_filtered_nr.append(df [df["overlap_frac_of_nr_in_endo"] > min_overlap]) # at least 50% of the NR area needs to overlap with an endosome
    overlap_endo_filtered_endo.append(df [df["overlap_frac_of_endo_has_nr"] > min_overlap]) # at least 50% of the endosome area needs to overlap with a NR

for df in overlap_df_lyso:
    
    overlap_lyso_filtered_nr.append(df [df["overlap_frac_of_nr_in_lyso"] > min_overlap]) # at least 50% of the NR area needs to overlap with a lysosome
    overlap_lyso_filtered_lyso.append(df[df["overlap_frac_of_lyso_has_nr"] > min_overlap]) # at least 50% of the lysosome area needs to overlap with a NR

After extracting the data, the number of positives must be counted for each cell to add the information to the original dataframe.

In [ ]:
# POSITIVE NR VESICLES FOR ENDO AND LYSOSOME
positive_per_cell_endo = []

for df in overlap_endo_filtered_nr:
    positive_per_cell_endo.append(
    df
    .groupby(["cell_uid"]) # Group by the unique cell ID to assign each value to the right cell
    .size()
    .reset_index(name="positive_nr_vesicles_endo")
    )

positive_per_cell_lyso = []

for df in overlap_lyso_filtered_nr:
    positive_per_cell_lyso.append(
    df
    .groupby(["cell_uid"])
    .size()
    .reset_index(name="positive_nr_vesicles_lyso")
)

# POSITIVE ENDO AND LYSO VESICLES FOR NRS

positive_per_cell_endo_has_nr = []

for df in overlap_endo_filtered_endo:
    positive_per_cell_endo_has_nr.append(
    df
    .groupby(["cell_uid"])
    .size()
    .reset_index(name="positive_endo_vesicles")
)

positive_per_cell_lyso_has_nr = []

for df in overlap_lyso_filtered_lyso:
    positive_per_cell_lyso_has_nr.append(
    df
    .groupby(["cell_uid"])
    .size()
    .reset_index(name="positive_lyso_vesicles")
)

Now, the counted vesicles can be added to the original dataframe

In [ ]:
df_complete = experiments.copy() # copy to keep the original data

# make a list with all dfs that will be added
additional_dfs = [
    positive_per_cell_endo,
    positive_per_cell_lyso,
    positive_per_cell_endo_has_nr,
    positive_per_cell_lyso_has_nr
]

#
for additions in additional_dfs:
    for i, (e, df) in enumerate(zip(df_complete, additions)):
        df_complete[i] = e.merge(
            df,
            on="cell_uid",
            how="left"
        )

Some cells will not have any overlapping vesicles. These were added to the df as N/A. They need to be set to 0.

In [ ]:
for e in df_complete:
    e["positive_nr_vesicles_endo"] = (
        e["positive_nr_vesicles_endo"]
    .fillna(0)
    .astype(int)
    )

    e["positive_nr_vesicles_lyso"] = (
    e["positive_nr_vesicles_lyso"]
    .fillna(0)
    .astype(int)
    )

    e["positive_endo_vesicles"] = (
    e["positive_endo_vesicles"]
    .fillna(0)
    .astype(int)
    )

    e["positive_lyso_vesicles"] = (
    e["positive_lyso_vesicles"]
    .fillna(0)
    .astype(int)
    )

    e = e.drop("List for coloc per vesicle endo", axis=1) # drop the json list 
    e = e.drop("List for coloc per vesicle lyso", axis=1)

Last but not least, the fraction of vesices that are overlapping can be calculated. As described above, for both endosomes and lysosomes the fractions can be calculated in two directions.

In [ ]:
for e in df_complete:
    e["fraction_nr_in_endo"] = (
        e["positive_nr_vesicles_endo"] / e["NR vesicle count"]
    )
    
    e["fraction_nr_in_lyso"] = (
        e["positive_nr_vesicles_lyso"] / e["NR vesicle count"]
    )
    
    e["fraction_endo_with_nr"] = (
        e["positive_endo_vesicles"] / e["Endo vesicle count"]
    )
    e["fraction_lyso_with_nr"] = (
        e["positive_lyso_vesicles"] / e["Lyso vesicle count"]
    )

exp_4_complete, exp_5_complete, exp_6_complete = df_complete # extract single dfs from list

#save extracted data
exp_4_complete.to_csv(r"YOUR_PATH\Exp_4\Data_extracted.csv")
exp_5_complete.to_csv(r"YOUR_PATH\Exp_5\Data_extracted.csv")
exp_6_complete.to_csv(r"YOUR_PATH\Exp_6\Data_extracted.csv")

For statistical analyses and making plots in seaborn, it makes sense to generate a single dataframe with all biological replicates. For that, each experiment needs to be assigned a biological replicate.

In [ ]:
# assign biological replicates
exp_4_complete["biorep"] = 1
exp_5_complete["biorep"] = 2
exp_6_complete["biorep"] = 3

# combine dataframes
df_statistics = pd.concat([exp_4_complete, exp_5_complete, exp_6_complete])